We will use `ERA5-land` dataset to enhance the NOAA Storm events data. A lot of events in `StormEvents_2014_2024.csv` have information about when and where they started and ended. We use this information to get metereological data at the space-time midpoint.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
import seaborn as sns
from statsmodels.tsa.stattools import grangercausalitytests, ccf
import numpy as np
import plotly.express as px
from datetime import datetime
import geopandas as gpd
import warnings
import xarray as xr
from datetime import datetime, timedelta

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
from utils import *

We read the `StormEvents_2014_2024.csv` data.

In [2]:
df_events = pd.read_csv("../data/NOAA_StormEvents/StormEvents_2014_2024.csv")
df_events

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,201402,18,1000,201402,18,2000,83473,503953,NEW HAMPSHIRE,33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low pressure developing south of Long Island a...,Eight to twelve inches of snow fell across eas...,CSV
1,201403,30,831,201403,30,931,83971,507163,MASSACHUSETTS,25,...,1.0,WNW,CHELMSFORD CENTER,42.5861,-71.3472,42.5867,-71.3469,A stacked low pressure system passed south and...,Boston Road was closed near Brian Road due to ...,CSV
2,201404,27,2306,201404,27,2306,83517,506236,MISSOURI,29,...,1.0,W,AVA,36.9500,-92.6600,36.9500,-92.6600,A powerful storm system and a dry line produce...,NaN,CSV
3,201404,27,2303,201404,27,2303,83517,506237,MISSOURI,29,...,1.0,W,AVA,36.9500,-92.6600,36.9500,-92.6600,A powerful storm system and a dry line produce...,Several power poles snapped and trees blown down.,CSV
4,201402,15,1300,201402,15,2100,83132,501499,WASHINGTON,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A strong cold front produced strong winds for ...,Two stations measured strong wind gusts in the...,CSV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
691429,202405,26,1148,202405,26,1148,192532,1188957,KENTUCKY,21,...,1.0,E,EAST NEEDMORE,37.6500,-84.7200,37.6500,-84.7200,A strong storm system moved across the Ohio an...,A trained spotter estimated 60 mph wind gusts ...,CSV
691430,202405,22,1809,202405,22,1809,192530,1188234,INDIANA,18,...,2.0,ESE,SCOTTSBURG,38.6713,-85.7364,38.6713,-85.7364,A cold front moved into the Ohio Valley during...,A tree was down at Lovers Lane and Prewitt Lane.,CSV
691431,202405,22,1757,202405,22,1757,192530,1188232,INDIANA,18,...,2.0,NE,CANNELTON,37.9478,-86.7247,37.9478,-86.7247,A cold front moved into the Ohio Valley during...,A tree was reported down over Chestnut Grove R...,CSV
691432,202406,23,1745,202406,23,1750,191388,1192879,NEW HAMPSHIRE,33,...,1.0,NE,GREENLAND,43.0400,-70.8400,43.0400,-70.8400,A supercell thunderstorm developed across sout...,A supercell thunderstorm dropped hail the size...,CSV


We open `ERA5-land` data as an `xarray` DataSet and store it in `ds`. This `.grib` file was downloaded from the [Copernicus Climate Data Store](https://cds.climate.copernicus.eu/), see `CDSdownload.ipynb`.

In [14]:
grib_file_path = '../Data/ERA5_land/reanalysis-era5-land-2014.grib'
ds = xr.open_dataset(grib_file_path, engine='cfgrib')
ds

<xarray.Dataset> Size: 5GB
Dimensions:     (time: 366, step: 4, latitude: 261, longitude: 591)
Coordinates:
    number      int32 4B ...
  * time        (time) datetime64[ns] 3kB 2013-12-31 2014-01-01 ... 2014-12-31
  * step        (step) timedelta64[ns] 32B 06:00:00 12:00:00 ... 1 days 00:00:00
    surface     float64 8B ...
  * latitude    (latitude) float64 2kB 50.0 49.9 49.8 49.7 ... 24.2 24.1 24.0
  * longitude   (longitude) float64 5kB -125.0 -124.9 -124.8 ... -66.1 -66.0
    valid_time  (time, step) datetime64[ns] 12kB ...
Data variables:
    t2m         (time, step, latitude, longitude) float32 903MB ...
    u10         (time, step, latitude, longitude) float32 903MB ...
    v10         (time, step, latitude, longitude) float32 903MB ...
    sp          (time, step, latitude, longitude) float32 903MB ...
    tp          (time, step, latitude, longitude) float32 903MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-03-19T17:14 GRIB to CDM+CF via cfgrib-0.9.1...

In this example we filter out the Storm events corresponding to the year 2014 (`df2014`) to reduce the runtime.

In [266]:
df2014 = df_events[df_events['END_YEARMONTH']<201501].copy()
#df2014 = df_events.sample(100).copy() # for testing
df2014

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,201402,18,1000,201402,18,2000,83473,503953,NEW HAMPSHIRE,33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low pressure developing south of Long Island a...,Eight to twelve inches of snow fell across eas...,CSV
1,201403,30,831,201403,30,931,83971,507163,MASSACHUSETTS,25,...,1.0,WNW,CHELMSFORD CENTER,42.5861,-71.3472,42.5867,-71.3469,A stacked low pressure system passed south and...,Boston Road was closed near Brian Road due to ...,CSV
2,201404,27,2306,201404,27,2306,83517,506236,MISSOURI,29,...,1.0,W,AVA,36.9500,-92.6600,36.9500,-92.6600,A powerful storm system and a dry line produce...,NaN,CSV
3,201404,27,2303,201404,27,2303,83517,506237,MISSOURI,29,...,1.0,W,AVA,36.9500,-92.6600,36.9500,-92.6600,A powerful storm system and a dry line produce...,Several power poles snapped and trees blown down.,CSV
4,201402,15,1300,201402,15,2100,83132,501499,WASHINGTON,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A strong cold front produced strong winds for ...,Two stations measured strong wind gusts in the...,CSV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59470,201405,1,0,201405,31,2359,85830,518264,NEW MEXICO,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Another very dry month led to some increase in...,The zone remained in Severe(D2) drought throug...,CSV
59471,201405,1,0,201405,31,2359,85830,518261,NEW MEXICO,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Another very dry month led to some increase in...,The zone remained in Severe(D2) drought throug...,CSV
59472,201402,21,1313,201402,21,1313,92828,556773,VIRGINIA,51,...,2.0,NW,DELTAVILLE,37.5700,-76.3500,37.5700,-76.3500,Scattered severe thunderstorms in advance of a...,Trees were downed.,CSV
59473,201402,21,1310,201402,21,1310,92828,556763,VIRGINIA,51,...,0.0,N,ONEMO,37.4000,-76.2700,37.4000,-76.2700,Scattered severe thunderstorms in advance of a...,Numerous trees were snapped.,CSV


We define `get_ds_mean`, a function that takes the initial and final space-time of a storm event and return the metereological data from `ds`.

In [ ]:
def get_ds_mean(t0,x0,y0,t1,x1,y1,ds):
    """
    Parameters:
    -t0,x0,y0: start time, latitude, longitude
    -t1,x1,y1: end time, latitude, longitude
    -ds: xarray dataset
    Returns:
    A list containing the features of ds at the midpoint of the time interval [t0,t1] and the spatial interval [(x0,y0),(x1,y1)]
    """
    features = list(ds.keys())
    if not any(np.isnan([x0, y0 , x1, y1])):
        t00 = datetime.strptime(t0,"%d-%b-%y %H:%M:%S")
        t11 = datetime.strptime(t1,"%d-%b-%y %H:%M:%S")
        tm = datetime.strftime(t00 + (t11 - t00)/2,"%d-%b-%y %H:%M:%S")
        t = tm.split(" ")[0]
        h = tm.split(" ")[1]
        x = (x0 + x1)/2
        y = (y0 + y1)/2
        
        temp = ds.sel(time=t, step=h, latitude=x, longitude=y, method='nearest')
        return [temp[key].values for key in features]
    else:
        return [np.nan for key in features]


features = list(ds.keys())
df2014[features] = pd.DataFrame(df2014.apply(lambda x:get_ds_mean(x['BEGIN_DATE_TIME'],x['BEGIN_LAT'],x['BEGIN_LON'],x['END_DATE_TIME'],x['END_LAT'],x['END_LON'],ds),axis=1).tolist(),index=df2014.index) #takes about 24 minutes to run through a year of df


In [265]:
df2014

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,t2m,u10,v10,sp,tp
465876,202112,10,1857,202112,10,1857,164477,993005,MISSOURI,29,...,37.3400,-93.7000,The Ozarks were on the western edge of a sever...,Golf ball sized hail was reported.,CSV,269.81622,0.83566284,-0.0033569336,99451.72,0.0001621306
134434,201601,21,2200,201601,22,1700,102458,612322,KENTUCKY,21,...,NaN,NaN,Western Kentucky was heavily impacted by the w...,NaN,CSV,NaN,NaN,NaN,NaN,NaN
662695,202408,5,1200,202408,5,1800,196294,1214757,SOUTH DAKOTA,46,...,NaN,NaN,Areas ahead of a weak frontal boundary exhibit...,Afternoon heat index values reached 100 degree...,CSV,NaN,NaN,NaN,NaN,NaN
470321,202107,17,1333,202107,17,1333,160544,973028,PENNSYLVANIA,42,...,40.4100,-79.5000,Showers and thunderstorms formed ahead of a cr...,Tree damage was reported in Salem Township.,CSV,267.04828,3.25029,-0.21690369,98415.38,2.9414891e-06
640554,202407,10,1625,202407,10,1625,192201,1192421,NEW YORK,36,...,42.9300,-75.8500,As a warm front lifted north across the area i...,Strong thunderstorm winds knocked down trees i...,CSV,266.28107,3.8278503,-0.16156006,96650.72,0.00026395023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325289,201903,8,300,201903,8,300,133757,811427,NEVADA,32,...,NaN,NaN,A slow moving and somewhat moist winter storm ...,"Received 8 inches at Austin Coop, elevation 71...",CSV,NaN,NaN,NaN,NaN,NaN
21179,201401,21,1900,201401,22,1200,82575,497376,NORTH CAROLINA,37,...,NaN,NaN,In the wake of arctic cold front moving east t...,Snow amounts averaged between 2 to 3.5 inches ...,CSV,NaN,NaN,NaN,NaN,NaN
596209,202306,23,2015,202306,23,2018,180125,1098460,TEXAS,48,...,33.1902,-101.3678,Strong daytime heating and good moisture level...,A Texas Tech University West Texas mesonet sit...,CSV,266.97052,-2.2346497,-1.6498413,93611.72,0.001168257
585699,202309,16,1100,202309,16,1900,185999,1141662,ARIZONA,4,...,NaN,NaN,Hot temperatures 3 to 5 degrees above normal c...,Hot temperatures 3 to 5 degrees above normal c...,CSV,NaN,NaN,NaN,NaN,NaN


Now `df2014` has 4 new columns,
- `t2m` : 2m temperature
- `u10` : 10m u-component of wind
- `v10` : 10m v-component of wind
- `sp` : surface pressure
- `tp` : total precipitation

Try with `eaglei_outages_with_county_info.parquet`

In [4]:
df_eaglei = pd.read_parquet("../data/eaglei_data/eaglei_outages_with_county_info.parquet", engine='pyarrow')
df_eaglei

,fips_code,customers_out,datetime,YEAR,NAME,STUSPS,FIPS,Pct_Buried_Lines,neighbors,Subregion,centroid_longitude,centroid_latitude,centroid_rounded,POPULATION,BUILDVALUE,AGRIVALUE,AREA,SOVI_SCORE,Power_Dependent_Devices_DME_Mean
0,1001.0,2.083333,2019-01-01 00:00:00,2019,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,610.083333
1,1001.0,0.000000,2019-01-01 06:00:00,2019,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,610.083333
2,1001.0,0.125000,2019-01-01 12:00:00,2019,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,610.083333
3,1001.0,0.958333,2019-01-01 18:00:00,2019,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,610.083333
4,1001.0,0.000000,2019-01-02 00:00:00,2019,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,610.083333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36082205,56029.0,329.000000,2014-11-29 06:00:00,2014,Park,WY,56029,0.205586,"[56039, 30031, 30009, 56013, 56003, 30067, 560...",NWPP,-109.50,44.5,POINT (-109.5 44.5),28205.0,3.265297e+09,85173000.0,6939.315034,36.775719,636.166667
36082206,56029.0,329.000000,2014-11-29 12:00:00,2014,Park,WY,56029,0.205586,"[56039, 30031, 30009, 56013, 56003, 30067, 560...",NWPP,-109.50,44.5,POINT (-109.5 44.5),28205.0,3.265297e+09,85173000.0,6939.315034,36.775719,636.166667
36082207,56029.0,329.000000,2014-11-29 18:00:00,2014,Park,WY,56029,0.205586,"[56039, 30031, 30009, 56013, 56003, 30067, 560...",NWPP,-109.50,44.5,POINT (-109.5 44.5),28205.0,3.265297e+09,85173000.0,6939.315034,36.775719,636.166667
36082208,56029.0,225.750000,2014-11-30 00:00:00,2014,Park,WY,56029,0.205586,"[56039, 30031, 30009, 56013, 56003, 30067, 560...",NWPP,-109.50,44.5,POINT (-109.5 44.5),28205.0,3.265297e+09,85173000.0,6939.315034,36.775719,636.166667


In [12]:
df2014_eaglei_dec = df_eaglei[(df_eaglei["YEAR"]==2014) & (df_eaglei["datetime"].dt.month ==12) ]
df2014_eaglei_dec

,fips_code,customers_out,datetime,YEAR,NAME,STUSPS,FIPS,Pct_Buried_Lines,neighbors,Subregion,centroid_longitude,centroid_latitude,centroid_rounded,POPULATION,BUILDVALUE,AGRIVALUE,AREA,SOVI_SCORE,Power_Dependent_Devices_DME_Mean
35651021,1001.0,0.000000,2014-12-01 00:00:00,2014,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,556.083333
35651022,1001.0,0.000000,2014-12-01 06:00:00,2014,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,556.083333
35651023,1001.0,0.000000,2014-12-01 12:00:00,2014,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,556.083333
35651024,1001.0,0.000000,2014-12-01 18:00:00,2014,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,556.083333
35651025,1001.0,0.041667,2014-12-02 00:00:00,2014,Autauga,AL,1001,0.079844,"[1051, 1085, 1101, 1047, 1021]",SRSO,-86.75,32.5,POINT (-86.75 32.5),54571.0,5.075584e+09,21460000.0,594.448312,25.857312,556.083333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36082200,55141.0,0.375000,2014-12-27 18:00:00,2014,Wood,WI,55141,0.291664,"[55057, 55001, 55019, 55097, 55053, 55073]",MROE,-90.00,44.5,POINT (-90 44.5),74749.0,9.952477e+09,141094000.0,793.058781,34.804888,571.333333
36082201,55141.0,0.000000,2014-12-28 00:00:00,2014,Wood,WI,55141,0.291664,"[55057, 55001, 55019, 55097, 55053, 55073]",MROE,-90.00,44.5,POINT (-90 44.5),74749.0,9.952477e+09,141094000.0,793.058781,34.804888,571.333333
36082202,55141.0,0.000000,2014-12-28 06:00:00,2014,Wood,WI,55141,0.291664,"[55057, 55001, 55019, 55097, 55053, 55073]",MROE,-90.00,44.5,POINT (-90 44.5),74749.0,9.952477e+09,141094000.0,793.058781,34.804888,571.333333
36082203,55141.0,0.208333,2014-12-28 12:00:00,2014,Wood,WI,55141,0.291664,"[55057, 55001, 55019, 55097, 55053, 55073]",MROE,-90.00,44.5,POINT (-90 44.5),74749.0,9.952477e+09,141094000.0,793.058781,34.804888,571.333333


In [45]:
test = df2014_eaglei_dec["datetime"].values[0]
np.datetime_as_string(test,unit='s')

'2014-12-01T00:00:00'

In [72]:
sample = df2014_eaglei_dec.sample(1)
sampleinfo = sample[["datetime","centroid_longitude","centroid_latitude"]].values[0]
str(sampleinfo[0])
sampleinfo
#get_ds(sampleinfo[0],sampleinfo[1],sampleinfo[2],ds)

array([Timestamp('2014-12-06 18:00:00'), -84.5, 34.0], dtype=object)

In [ ]:
ds.sel(time='2014-12-06', step='18:00:00', latitude='')

In [61]:
def get_ds(t,x,y,ds):
    """
    Parameters:
    -t,x,y: start time, latitude, longitude
    -ds: xarray dataset
    Returns:
    A list containing the features of ds at the midpoint of the time interval [t0,t1] and the spatial interval [(x0,y0),(x1,y1)]
    """
    features = list(ds.keys())
    if not any(np.isnan([x, y])):
        t0 = str(t)
        tt = t0.split(" ")[0]
        hh = t0.split(" ")[1]
        
        temp = ds.sel(time=tt, step=hh, latitude=x, longitude=y, method='nearest')
        return [temp[key].values for key in features]
    else:
        return [np.nan for key in features]

In [ ]:
features = list(ds.keys())
df2014_eaglei_dec[features] = pd.DataFrame(df2014_eaglei_dec.apply(lambda x:get_ds(x['datetime'],x['centroid_latitude'],x['centroid_longitude'],ds),axis=1).tolist(),index=df2014_eaglei_dec.index) #takes about 24 minutes to run through a year of df

IndexError: list index out of range